# DSI490 — Flood Agent · Notebook สำรวจ v1.3

Schema, Extraction, Backend Rules และ Agent ใช้ไฟล์จาก `dspy-agent/app/` ชุดเดียวกับ API
Notebook นี้ใช้สำรวจและทดลอง ไม่คัดลอก prompt หรือกฎตัดสินใจมาแยกแก้
รันเซลล์ที่เรียกโมเดลจะใช้ Gemini จริงและมีค่าใช้จ่าย ควรใช้ข้อมูลสมมติ


In [ ]:
import os
import sys
from pathlib import Path
import dspy
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dspy-agent").is_dir())
APP_ROOT = ROOT / "dspy-agent"
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))
load_dotenv(APP_ROOT / ".env", override=False)
load_dotenv(ROOT / "lab" / "1" / ".env", override=False)  # รองรับไฟล์ key เดิมของ Notebook


In [ ]:
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or os.getenv("API_KEY")
model = os.getenv("GEMINI_MODEL", "gemini/gemini-3.1-flash-lite")
if not api_key:
    raise RuntimeError("ตั้ง GEMINI_API_KEY ใน dspy-agent/.env หรือ lab/1/.env ก่อนเรียก Gemini")
lm = dspy.LM(model=model, api_key=api_key)
dspy.configure(lm=lm)
print("model =", model)


In [ ]:
import json
from app.agent import FloodAgent
from app.schemas import Hotline, SCHEMA_VERSION
from app.repository import Repository
from app.knowledge import KnowledgeBase
from app.areas import AreaResolver

# โหลด config เดียวกับ API; ห้ามสร้าง verified_at ใหม่เพื่อแสร้งว่าตรวจแล้ว
hotline_config = json.loads((APP_ROOT / "app" / "hotlines_th.json").read_text(encoding="utf-8"))
HOTLINES = [Hotline.model_validate(row) for row in hotline_config["hotlines"]]
print("schema =", SCHEMA_VERSION, "| hotline review =", hotline_config.get("status"))


def make_agent():
    # Notebook จำลองการส่งคำตอบสำเร็จ และแยกข้อมูลจากฐานข้อมูล API
    # ไม่มี sender: รายงานอยู่ใน outbox เป็น pending ไม่ส่ง network ไป Agent 2
    return FloodAgent(
        hotlines=HOTLINES,
        repository=Repository("sqlite:///:memory:"),
        knowledge=KnowledgeBase(APP_ROOT / "app" / "knowledge_th.csv"),
        area_resolver=AreaResolver(APP_ROOT / "app" / "areas_singburi.csv"),
        internal_user_id="usr_notebook_demo",
    )

import re
from app.schemas import *
print("ROOT =", ROOT)


## A. สำรวจ Schema

ลองสร้าง `IncidentState` จากข้อความ "น้ำเข้าบ้านที่ตำบลทับยา อินทร์บุรี น้ำถึงเอว มีคนแก่สองคน แค่อยากแจ้งไว้ก่อน" (Scenario G02)
สังเกตว่า field ที่ยังไม่ได้ถามมีสถานะ `not_asked` ไม่ใช่ค่าว่าง

In [ ]:
from app.schemas import TrackedField as TF

state = IncidentState(
    internal_user_id="usr_demo",
    conversation_id="conv_demo",
    # ข้อมูลพื้นที่ฉบับร่างยังรอทีมตรวจ จึงไม่อ้างว่า matched (v1.3 ข้อ 6.4)
    location_area=TF[AreaRef](status=FieldStatus.KNOWN, updated_at=now_bkk(), value=AreaRef(
        raw="ตำบลทับยา อำเภออินทร์บุรี", stated_level=AreaLevel.TAMBON, match_status=AreaMatchStatus.UNVERIFIED)),
    situation=TF[Situation](status=FieldStatus.KNOWN, value=Situation(text="น้ำเข้าบ้าน", tags=[SituationTag.WATER_IN_HOUSE])),
    needs=TF[list[Need]](status=FieldStatus.KNOWN, value=[Need.INFO_ONLY]),
    water_level=TF[WaterLevel](status=FieldStatus.KNOWN, value=WaterLevel(raw="ถึงเอว", descriptor=WaterDescriptor.WAIST)),
    vulnerable_people=TF[VulnerablePeople](
        status=FieldStatus.KNOWN,
        value=VulnerablePeople(present=True, groups=[VulnerableGroup(type=VulnerableType.ELDERLY, count=2)]),
    ),
)

for name in IncidentFieldName:
    f = state.tracked_field(name)
    if f is not None:
        print(f"{name.value:22} {f.status.value:10} {f.value if f.value is not None else ''}")

ตัวอย่างการคำนวณ field ที่ยังต้องถาม (เขียนไว้ดูภาพเท่านั้น — ของจริงจะอยู่ใน `memory.py` ขั้นที่ 3)

In [ ]:
still_to_ask = [n.value for n in (*REQUIRED_FIELDS, *ASK_ONCE_FIELDS) if state.tracked_field(n).can_ask]
print("ยังต้องถาม:", still_to_ask)

Schema ปฏิเสธข้อมูลที่ขัดกับกฎทันที เช่น บอกว่า `known` แต่ไม่มีค่า หรือ draft ที่มีคนติดค้างกลายเป็น `abandoned`

In [ ]:
from pydantic import ValidationError

for label, build in [
    ("known แต่ไม่มีค่า", lambda: TF[str](status=FieldStatus.KNOWN)),
    ("draft เร่งด่วนเป็น abandoned", lambda: IncidentState(internal_user_id="u", conversation_id="c",
        status=IncidentStatus.ABANDONED,
        urgent_flags=[UrgentFlagRecord(flag=UrgentFlag.PEOPLE_TRAPPED, sources=[UrgentFlagSource.KEYWORD], detected_at=now_bkk())])),
]:
    try:
        build()
    except ValidationError as e:
        print(f"✗ {label}: {e.errors()[0]['msg']}")

In [ ]:
print("field ใน IncidentReport (สิ่งที่ Agent 2 เห็น):")
print(", ".join(IncidentReport.model_fields))
print("\nมี LINE User ID / เบอร์ / ชื่อ ไหม:",
      bool({"line_user_id", "phone", "name", "internal_user_id"} & set(IncidentReport.model_fields)))

## B. ทดลองสนทนาผ่าน FloodAgent ตัวจริง

`ChatSession` เป็นเพียงตัวช่วยแสดงผล กฎทั้งหมดอยู่ใน `app/` เช่นเดียวกับ API
โหลดสายด่วนจาก config กลาง ใช้ฐานข้อมูลจำลอง ไม่มี sender ไป Agent 2


In [ ]:
class ChatSession:
    """UI wrapper สำหรับ Notebook; ไม่มีกฎรับแจ้งหรือ prompt แยกจากแอป"""
    def __init__(self):
        self.agent = make_agent()

    def say(self, message: str) -> str:
        from app.safety import redact_identifiers, redact_phones
        out = self.agent(input_text=message)
        safe_message = redact_identifiers(redact_phones(message).text)
        print(f"👤 {safe_message}\n🤖 {out.output_text}\n")
        return out.output_text


In [ ]:
chat = ChatSession()
chat.say("สวัสดีครับ")
chat.say("น้ำท่วมบ้านครับ")
chat.say("ตำบลทับยา อินทร์บุรีครับ")
chat.say("ประมาณ 80 เซนติเมตร มีผู้สูงอายุสองคน")
chat.say("เดี๋ยวก่อน ต้องตัดไฟตอนไหนครับ");

ตรวจเบื้องต้นด้วยเกณฑ์จาก Golden Scenarios: ห้ามอ้างว่าส่งความช่วยเหลือแล้ว (G06, G15) และห้ามบอกตัวเลขระดับน้ำในพื้นที่ (G11)

In [ ]:
GOLDEN = json.loads((ROOT / "dspy-agent" / "tests" / "golden_scenarios.json").read_text(encoding="utf-8"))
FORBIDDEN = GOLDEN["conventions"]["forbidden_claims"]
NUMBER_RE = re.compile(r"\d+\s*(ซม|เซน|cm|เมตร|ม\.)")

def check(text: str, *, no_numbers=False, must_contain=()):
    problems = [f"อ้างว่า: {c}" for c in FORBIDDEN if c in text]
    if no_numbers and NUMBER_RE.search(text):
        problems.append("มีตัวเลขระดับน้ำ")
    problems += [f"ไม่มี {s}" for s in must_contain if s not in text]
    print("✓ ผ่าน" if not problems else "✗ " + "; ".join(problems))

# G11 — ถามระดับน้ำในพื้นที่
t = ChatSession().say("ตอนนี้น้ำที่อินทร์บุรีสูงเท่าไหร่แล้วครับ")
check(t, no_numbers=True)

# G06 — มีคนติดค้าง
t = ChatSession().say("ช่วยด้วยครับ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก อยู่หมู่ 3 ตำบลน้ำตาล")
check(t, must_contain=("1784", "1669"))

## C. ทดลอง Extraction แยกจากกฎตัดสินใจ

นำเข้า `Extractor` และ prompt จาก `app/extraction.py` โดยตรง ทุก field ใช้ `schemas.py`
ผล `ExtractionResult` เป็นข้อมูลที่ดึงได้ ไม่ใช่รายงานที่ส่งให้ Agent 2 และไม่ใช่การประเมิน priority


In [ ]:
from app.extraction import Extractor, ExtractIntentAndSlots, INSTRUCTIONS
from app.safety import redact_identifiers, redact_phones

extractor = Extractor()


def run_extract(message, state=None, pending=None, history=None) -> ExtractionResult:
    # ทดลองเฉพาะ Extraction; context ครบ 5 ชั้นสร้างโดย FloodAgent เมื่อใช้ตัวจริง
    safe_message = redact_phones(redact_identifiers(message)).text
    return extractor(safe_message, state, pending)


def show(message, r: ExtractionResult):
    safe_message = redact_identifiers(redact_phones(message).text)
    print(f"👤 {safe_message}")
    print(f"   intents={[i.value for i in r.intents]} target={r.target} location={r.mentioned_location}")
    for update in r.updates:
        print(f"   • {update.field.value}: {update.status.value} {json.dumps(update.value, ensure_ascii=False)}")
    if r.urgent_flags:
        print("   urgent_flags:", [flag.value for flag in r.urgent_flags])
    if r.question:
        print("   question:", r.question)


In [ ]:
SAMPLES = [
    "น้ำท่วมบ้านครับ",
    "น้ำเข้าบ้านที่ตำบลทับยา อินทร์บุรี น้ำถึงเอว มีคนแก่สองคน แค่อยากแจ้งไว้ก่อนครับ",
    "น้ำท่วมถนนเข้าหมู่บ้าน รถผ่านไม่ได้",
    "ไม่แน่ใจครับ ไม่รู้ว่าบ้านอื่นเป็นยังไง",
    "ช่วยด้วยครับ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก",
    "เดี๋ยวก่อน ต้องตัดไฟตอนไหนครับ",
    "มีผู้สูงอายุสองคนในบ้าน",
    "น้ำท่วมหน้าวัดโพธิ์ชัย ตำบลถอนสมอ",
    "บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ",
]
for m in SAMPLES:
    show(m, run_extract(m))

### คำตอบสั้นที่ต้องพึ่ง context (Scenario G03, G04, G09)

ส่ง `state` และ `pending_question` เข้าไปด้วย ดูว่า LLM ตีความถูกไหม

In [ ]:
# G03 — ตอบสั้นหลังถูกถามระดับน้ำ
show("80 เซนติเมตร", run_extract("80 เซนติเมตร", state=state, pending=[IncidentFieldName.WATER_LEVEL]))

# G04 — แก้ไขข้อมูลเดิม
show("แก้เป็น 80 เซนติเมตรครับ พิมพ์ผิด", run_extract("แก้เป็น 80 เซนติเมตรครับ พิมพ์ผิด", state=state))

# G09 — ตำแหน่งใหม่ระหว่างมีเหตุเปิด → target ควรเป็น unclear
show("บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ", run_extract("บ้านแม่ที่พรหมบุรีน้ำเข้าแล้วครับ", state=state))

## D. ตรวจ Extraction กับ Golden Scenarios

ตรวจเฉพาะ intent/field/urgent flag จาก turn แรก ผลจาก Gemini อาจเปลี่ยนได้และต้องทดลองจริง
ส่วน lifecycle, consent, events, priority และการส่งต่อทดสอบใน `dspy-agent/tests/`
ตัวเลขผลด้านล่างเป็นผลที่ได้จากการรันครั้งนั้น ไม่ใช่ผลเทสต์ offline ของแอป


In [ ]:
def _check_field(name, spec, update):
    """ตรวจค่าของ field หนึ่งตาม expect ใน golden_scenarios.json คืนรายการปัญหา"""
    problems = []
    want_status = spec.get("status")
    if want_status in ("known", "unknown", "declined"):
        if update is None or update.status.value != want_status:
            got = "ไม่มี" if update is None else update.status.value
            return [f"{name}: ต้องเป็น {want_status} แต่ได้ {got}"]
    if update is None or update.status != FieldStatus.KNOWN:
        return problems
    v = update.value
    if "value_contains" in spec and spec["value_contains"] not in str(v):
        problems.append(f"{name}: ไม่มีคำว่า {spec['value_contains']!r} (ได้ {v!r})")
    if "text_contains" in spec and spec["text_contains"] not in v.get("text", ""):
        problems.append(f"{name}.text: ไม่มีคำว่า {spec['text_contains']!r}")
    if "tags_include" in spec and (miss := set(spec["tags_include"]) - set(v.get("tags", []))):
        problems.append(f"{name}.tags: ขาด {sorted(miss)}")
    for key in ("value_include", "if_known_value_include"):
        if key in spec and (miss := set(spec[key]) - set(v)):
            problems.append(f"{name}: ขาด {sorted(miss)}")
    if "descriptor" in spec and v.get("descriptor") != spec["descriptor"]:
        problems.append(f"{name}.descriptor: ต้องเป็น {spec['descriptor']} แต่ได้ {v.get('descriptor')}")
    if "cm" in spec and v.get("cm") != spec["cm"]:
        problems.append(f"{name}.cm: ต้องเป็น {spec['cm']} แต่ได้ {v.get('cm')}")
    for g in spec.get("groups_include", []):
        if not any(all(grp.get(k) == val for k, val in g.items()) for grp in v.get("groups", [])):
            problems.append(f"{name}.groups: ไม่มี {g}")
    return problems


def grade(scenario):
    turn = scenario["turns"][0]
    exp = turn["expect"]
    r = run_extract(turn["user"])
    got_intents = {i.value for i in r.intents}
    got_flags = {f.value for f in r.urgent_flags}
    updates = {u.field.value: u for u in r.updates}
    problems = []
    if miss := set(exp.get("intents_include", [])) - got_intents:
        problems.append(f"intent ขาด {sorted(miss)}")
    if miss := set(exp.get("urgent_flags_include", [])) - got_flags:
        problems.append(f"urgent_flags ขาด {sorted(miss)}")
    if "urgent_flags" in exp and got_flags != set(exp["urgent_flags"]):
        problems.append(f"urgent_flags ต้องเป็น {sorted(exp['urgent_flags'])} แต่ได้ {sorted(got_flags)}")
    for name, spec in exp.get("fields", {}).items():
        problems += _check_field(name, spec, updates.get(name))
    return problems

rows = []
for s in GOLDEN["scenarios"]:
    if s["kind"] == "conversation" and "setup" not in s:
        problems = grade(s)
        rows.append((s["id"], s["title"], problems))
        print(f"{'✓' if not problems else '✗'} {s['id']} {s['title']}" + ("" if not problems else "\n    " + "\n    ".join(problems)))

passed = sum(not p for *_, p in rows)
print(f"\nผ่าน {passed}/{len(rows)}")

## E. ต่อสายผ่าน FloodAgent เช่นเดียวกับ API

```text
ปิดบังข้อมูล → โหลด context → Extraction → Python Rules → priority
→ บันทึก state + outbox → ส่งเคสด่วนก่อนเรียบเรียง
→ RAG/คำถามตามแผน → ตรวจถ้อยคำ → template สำรอง → บันทึกคำตอบ
```

ไม่มี MiniPipeline หรือ ASK_ORDER ที่ตัดสินใจแยกใน Notebook แล้ว
ตัวช่วยด้านล่างเรียก Agent ตัวเดียวกับระบบจริง แต่ใช้ฐานข้อมูลจำลองและไม่มี transport


In [ ]:
class PipelineDemo:
    """ตัวช่วยแสดงผลเท่านั้น; state และ pending question อยู่ใน FloodAgent"""
    def __init__(self):
        self.agent = make_agent()

    @property
    def state(self):
        return self.agent.state

    def turn(self, text: str):
        out = self.agent(input_text=text)
        print(out.output_text)
        self.agent.show_last()
        return out

    def show_state(self):
        self.agent.show_state()


In [ ]:
p = PipelineDemo()
p.turn("น้ำท่วมบ้านค่ะ")
p.turn("ตำบลทับยา อินทร์บุรีค่ะ แค่อยากแจ้งไว้ก่อน")
p.turn("ไม่ทราบจำนวนคนและกลุ่มเปราะบางค่ะ")
p.turn("80 เซนติเมตร")
p.turn("แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด")
p.turn("ยายติดอยู่บนหลังคา ช่วยด้วย เบอร์ 081-234-5678")
p.show_state()


สิ่งที่ควรสังเกต:
- `initial` / `correction` ตัดสินโดยโค้ด ไม่เชื่อ label จาก Gemini เพียงอย่างเดียว
- เหตุด่วนถูกบันทึกใน outbox ก่อนเรียบเรียงคำตอบ สถานะยังเป็น pending เพราะไม่มี sender
- เบอร์ถูกปิดบังในข้อความและไม่เข้ารายงาน ต้องยืนยันอนุญาตใช้เบอร์แยก
- ไม่เดา rescue จาก urgent flag และไม่ใช้การยืนยันรายงานแทน consent
- Priority เป็นกฎรายเคสฉบับร่างตามข้อ 7 ไม่แสดงให้ประชาชนเห็น


## ตรวจผลและงานที่ยังรอทีม

- รัน `python -m pytest -q dspy-agent/tests` จาก root สำหรับเทสต์ที่ไม่เรียกบริการจริง
- คำตอบ Gemini ใน Notebook ต้องทดสอบจริงแยกจาก unit tests
- ดูตัวอย่าง submission/update ที่ `dspy-agent/agent2_input_example.json` และ `agent2_update_example.json`
- รายชื่อพื้นที่/KB ต้องผ่านการตรวจจากทีมก่อนใช้อ้างอิง สัญญา Agent 1–2 และเกณฑ์ priority ยังเป็นร่าง
- รายละเอียดข้อจำกัดและการตั้งค่าอยู่ใน `dspy-agent/docs/v1.3-implementation.md`


In [ ]:
from pydantic import TypeAdapter

for filename in ("agent2_input_example.json", "agent2_update_example.json"):
    event = TypeAdapter(HandoffEvent).validate_json((APP_ROOT / filename).read_text(encoding="utf-8"))
    print(filename, "→", event.event_type.value, event.case_version)

print("Outbox:", [(r.event_type.value, r.delivery_status.value) for r in p.agent.repository.outbox()])
